In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
# 设置中文字体
matplotlib.rcParams['font.sans-serif'] = ['SimHei']   # 黑体
# 解决负号显示异常
matplotlib.rcParams['axes.unicode_minus'] = False

In [2]:

df_all = pd.read_feather("Dengue.feather")
df_all

,Time,CNNLSTMForecaster,Labels,City,Dataset,CNNForecaster,DLinearForecaster,LSTMForecaster,MLPForecaster,ResNetForecaster,TCNForecaster,TransformerForecaster,AutoformerForecaster,TimesNetForecaster,XGBSingleForecaster
0,2005-01-01,-23.729261,0.0,江西,train,-12.565220,-26.217857,-21.672554,-12.551907,-20.450478,-13.253731,-17.236217,-12.854452,-23.756239,-0.004233
1,2005-02-01,-24.912880,0.0,江西,train,-12.453150,-18.181519,-29.758297,-11.644766,-39.346741,-12.492916,-16.921816,-15.672018,-24.761414,-0.146540
2,2005-03-01,-23.615232,0.0,江西,train,-13.482731,-29.491089,-23.313293,-12.542044,-41.351574,-16.044102,-16.728395,-13.632906,-25.459473,0.057235
3,2005-04-01,-16.010069,0.0,江西,train,-12.565220,-29.656433,-28.960735,-12.551907,-19.052019,-13.253731,-11.154093,-12.920635,-22.945917,-0.029545
4,2005-05-01,-17.120281,0.0,江西,train,-12.453150,-24.433495,-26.937859,-11.644766,-38.081421,-12.492916,-12.138878,-15.757113,-20.853926,-0.125485
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5854,2020-05-01,-15.152290,0.0,西藏,test,-12.453150,9.763773,-15.155693,-11.644766,-34.356895,-12.492916,-18.020748,-15.656260,-31.681263,0.234179
5855,2020-06-01,-12.501762,0.0,西藏,test,-13.482731,3.468175,-9.986752,-12.542044,-14.911768,-16.044102,-14.951859,-13.576178,-32.363079,0.894556
5856,2020-07-01,-14.224161,0.0,西藏,test,-12.565220,28.857697,-26.218552,-12.551907,18.901398,-13.253731,-12.730844,-12.958456,-28.253147,9.329739
5857,2020-08-01,-8.608046,0.0,西藏,test,-12.453150,63.267071,-7.780233,-11.644766,12.008417,-12.492916,-13.819777,-15.694078,-28.080971,574.711792


In [105]:
df_all[['Dataset', 'City']].drop_duplicates()

,Dataset,City
0,train,江西
189,train,湖南
378,train,贵州
567,train,云南
756,train,浙江
945,train,海南
1134,train,福建
1323,train,江苏
1512,train,河北
1701,train,河南


In [122]:
CityName = '湖北'
DiseaseName = 'Dengue'
plt_data = df_all.query("City==@CityName")
DatasetName = plt_data['Dataset'].unique().tolist()[0]

plot_df = plt_data.rename(columns = {"Labels":"Ground truth"}).drop(columns = ['City', 'Dataset']).melt(
    id_vars=["Time"],
    value_name='Count',
    var_name='Model'
)


In [123]:
plot_df.groupby("Model").apply(lambda x: x.assign(Count = lambda y:(y['Count'] - y['Count'].mean())/ y['Count'].std()))

Time     Count
Model                                         
AutoformerForecaster 1701 2005-01-01  0.994524
                     1702 2005-02-01 -1.349731
                     1703 2005-03-01  0.372579
                     1704 2005-04-01  0.994524
                     1705 2005-05-01 -1.355049
...                              ...       ...
XGBSingleForecaster  2263 2020-05-01 -0.373783
                     2264 2020-06-01 -0.310591
                     2265 2020-07-01 -0.293884
                     2266 2020-08-01 -0.461613
                     2267 2020-09-01 -0.369010

[2268 rows x 2 columns]

In [124]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np 
# 保证时间列是 datetime
plot_df = plot_df.copy()

plot_df["Time"] = pd.to_datetime(plot_df["Time"])

color_map = {
    "Ground truth": "#D62728", # #D62728
    "CNNLSTMForecaster": "#1F77B4",
    "CNNForecaster": "#17BECF",
    "DLinearForecaster": "#2CA02C",
    "LSTMForecaster": "#9467BD",
    "MLPForecaster": "#FF7F0E",
    "ResNetForecaster": "#8C564B",
    "TCNForecaster": "#E377C2",
    "TransformerForecaster": "#7F7F7F",
    "XGBSingleForecaster": "#BCBD22", # BCBD22
}

dash_map = {
    "Ground truth": "dash",
    "CNNLSTMForecaster": "solid",
    "CNNForecaster": "solid",
    "DLinearForecaster": "solid",
    "LSTMForecaster": "solid",
    "MLPForecaster": "solid",
    "ResNetForecaster": "solid",
    "TCNForecaster": "solid",
    "TransformerForecaster": "solid",
    "XGBSingleForecaster": "solid",
}

model_order = [

    "CNNLSTMForecaster",
    "CNNForecaster",
    "DLinearForecaster",
    "LSTMForecaster",
    "MLPForecaster",
    "ResNetForecaster",
    "TCNForecaster",
    "TransformerForecaster",
    "XGBSingleForecaster",
    "Ground truth",
]

fig = go.Figure()

for model in model_order:
    sub_df = plot_df[plot_df["Model"] == model].sort_values("Time")
    if sub_df.empty:
        continue

    is_gt = model == "Ground truth"

    fig.add_trace(
        go.Scatter(
            x=sub_df["Time"],
            y=sub_df["Count"],
            mode="lines",
            name=model,
            visible=True if model in ["Ground truth", "TransformerForecaster", "XGBSingleForecaster"] else "legendonly",
            line=dict(
                color=color_map.get(model, "#444444"),
                width=3.8 if is_gt else 2.5,
                dash=dash_map.get(model, "solid"),
            ),
            opacity=.7 if is_gt else 1,
            hovertemplate=(
                f"<b>{model}</b><br>"
                "Time: %{x|%Y-%m}<br>"
                "Count: %{y:.3f}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    template="plotly_white",
    width=1400,
    height=520,
    hovermode="x unified",
    legend=dict(
        title="Model",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
        bgcolor="rgba(255,255,255,0.85)",
    ),
    margin=dict(l=50, r=30, t=70, b=60),
)

fig.update_xaxes(
    title_text="Time",
    showgrid=False,
    gridcolor="rgba(0,0,0,0.08)",
    showline=True,
    linecolor = 'grey',
    linewidth = 1.5,
    tickformat="%Y-%m",
    dtick="M3",  # 每3个月一个刻度，月度数据更清晰
    rangeslider=dict(visible=True),
    rangeselector=dict(
        x=0,
        xanchor="left",
        y=1.3,
        buttons=[
            dict(count=3, label="3m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            dict(count=12, label="1y", step="month", stepmode="backward"),
            dict(count=24, label="2y", step="month", stepmode="backward"),
            dict(step="all", label="All"),
        ]
    ),
)

fig.update_yaxes(
    title_text="Count",
    showgrid=True,
    showline=True,
    linecolor = 'grey',
    linewidth = 1.5,

    gridcolor="rgba(0,0,0,0.08)",
    zeroline=False,
)
fig.update_layout(
    title=dict(
        text=f"Monthly {DiseaseName} at {CityName} ({DatasetName} set)",
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    )
)


fig.show(config={
    "scrollZoom": True,
    "displaylogo": False,
    "doubleClick": "reset",
    "responsive": True,
})



In [125]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np 
# 保证时间列是 datetime
plot_df = plot_df.copy()
plot_df = plot_df.groupby("Model").apply(lambda x: x.assign(Count = lambda y:(y['Count'] - y['Count'].mean())/ y['Count'].std())).reset_index(drop=False)
plot_df["Time"] = pd.to_datetime(plot_df["Time"])

color_map = {
    "Ground truth": "#D62728", # #D62728
    "CNNLSTMForecaster": "#1F77B4",
    "CNNForecaster": "#17BECF",
    "DLinearForecaster": "#2CA02C",
    "LSTMForecaster": "#9467BD",
    "MLPForecaster": "#FF7F0E",
    "ResNetForecaster": "#8C564B",
    "TCNForecaster": "#E377C2",
    "TransformerForecaster": "#7F7F7F",
    "XGBSingleForecaster": "#BCBD22", # BCBD22
}

dash_map = {
    "Ground truth": "dash",
    "CNNLSTMForecaster": "solid",
    "CNNForecaster": "solid",
    "DLinearForecaster": "solid",
    "LSTMForecaster": "solid",
    "MLPForecaster": "solid",
    "ResNetForecaster": "solid",
    "TCNForecaster": "solid",
    "TransformerForecaster": "solid",
    "XGBSingleForecaster": "solid",
}

model_order = [

    "CNNLSTMForecaster",
    "CNNForecaster",
    "DLinearForecaster",
    "LSTMForecaster",
    "MLPForecaster",
    "ResNetForecaster",
    "TCNForecaster",
    "TransformerForecaster",
    "XGBSingleForecaster",
    "Ground truth",
]

fig = go.Figure()

for model in model_order:
    sub_df = plot_df[plot_df["Model"] == model].sort_values("Time")
    if sub_df.empty:
        continue

    is_gt = model == "Ground truth"

    fig.add_trace(
        go.Scatter(
            x=sub_df["Time"],
            y=sub_df["Count"],
            mode="lines",
            name=model,
            visible=True if model in ["Ground truth", "TransformerForecaster", "XGBSingleForecaster"] else "legendonly",
            line=dict(
                color=color_map.get(model, "#444444"),
                width=3.8 if is_gt else 2.5,
                dash=dash_map.get(model, "solid"),
            ),
            opacity=.7 if is_gt else 1,
            hovertemplate=(
                f"<b>{model}</b><br>"
                "Time: %{x|%Y-%m}<br>"
                "Count: %{y:.3f}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    template="plotly_white",
    width=1400,
    height=520,
    hovermode="x unified",
    legend=dict(
        title="Model",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
        bgcolor="rgba(255,255,255,0.85)",
    ),
    margin=dict(l=50, r=30, t=70, b=60),
)

fig.update_xaxes(
    title_text="Time",
    showgrid=False,
    gridcolor="rgba(0,0,0,0.08)",
    showline=True,
    linecolor = 'grey',
    linewidth = 1.5,
    tickformat="%Y-%m",
    dtick="M3",  # 每3个月一个刻度，月度数据更清晰
    rangeslider=dict(visible=True),
    rangeselector=dict(
        x=0,
        xanchor="left",
        y=1.3,
        buttons=[
            dict(count=3, label="3m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            dict(count=12, label="1y", step="month", stepmode="backward"),
            dict(count=24, label="2y", step="month", stepmode="backward"),
            dict(step="all", label="All"),
        ]
    ),
)

fig.update_yaxes(
    title_text="Count (normalized)",
    showgrid=True,
    showline=True,
    linecolor = 'grey',
    linewidth = 1.5,

    gridcolor="rgba(0,0,0,0.08)",
    zeroline=False,
)
fig.update_layout(
    title=dict(
        text=f"Monthly {DiseaseName} at {CityName} ({DatasetName} set)",
        x=0.5,
        xanchor="center",
        font=dict(size=20)
    )
)


fig.show(config={
    "scrollZoom": True,
    "displaylogo": False,
    "doubleClick": "reset",
    "responsive": True,
})



## 模型性能比较

In [115]:
# import numpy as np
# from sklearn.metrics import mean_squared_error, mean_absolute_error



# # ========== 2. 只取测试集 ==========
# df_test = df_all[df_all["Dataset"] == "test"].copy()

# # ========== 3. 模型列 ==========
# model_cols = [
#     "CNNLSTMForecaster",
#     "CNNForecaster",
#     "DLinearForecaster",
#     "LSTMForecaster",
#     "MLPForecaster",
#     "ResNetForecaster",
#     "TCNForecaster",
#     "TransformerForecaster",
#     "XGBSingleForecaster"
# ]

# # ========== 4. 定义评价函数 ==========
# def evaluate(y_true, y_pred):
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     mae = mean_absolute_error(y_true, y_pred)
    
#     # 防止除0
#     mask = y_true != 0
#     mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if np.any(mask) else np.nan
    
#     return rmse, mae, mape


# # ========== 5. 按省份评估 ==========
# results = []

# for city, group in df_test.groupby("City"):
#     y_true = group["Labels"].values
    
#     for model in model_cols:
#         y_pred = group[model].values
        
#         rmse, mae, mape = evaluate(y_true, y_pred)
        
#         results.append({
#             "City": city,
#             "Model": model,
#             "RMSE": rmse,
#             "MAE": mae,
#             "MAPE": mape
#         })

# # 转为DataFrame
# results_df = pd.DataFrame(results)

# # ========== 6. 每个省份内排序 ==========
# results_df["Rank_RMSE"] = results_df.groupby("City")["RMSE"].rank(method="min")
# results_df["Rank_MAE"] = results_df.groupby("City")["MAE"].rank(method="min")

# # ========== 7. 输出结果 ==========
# print("\n=== 每个省份模型表现 ===")
# print(results_df.sort_values(["City", "RMSE"]))

# # 保存
# results_df.to_csv("model_evaluation_by_city.csv", index=False)

# # ========== 8. 找每个省最优模型 ==========
# best_models = results_df.loc[
#     results_df.groupby("City")["RMSE"].idxmin()
# ]

# # print("\n=== 每个省份最优模型（按RMSE） ===")
# # print(best_models[["City", "Model", "RMSE"]])
# best_models


In [130]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# ========== 2. 只取测试集 ==========
df_test = df_all[df_all["Dataset"] == "test"].copy()

# ========== 3. 模型列 ==========
model_cols = [
    "CNNLSTMForecaster",
    "CNNForecaster",
    "DLinearForecaster",
    "LSTMForecaster",
    "MLPForecaster",
    "ResNetForecaster",
    "TCNForecaster",
    "TransformerForecaster",
    "XGBSingleForecaster"
]

# ========== 4. 爆发阈值 ==========
OUTBREAK_CUTOFF = 1.0

# ========== 5. 定义评价函数 ==========
def evaluate_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    # 防止除0
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if np.any(mask) else np.nan

    return rmse, mae, mape


import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    roc_curve
)

def evaluate_outbreak_detection(y_true, y_pred, cutoff=1.0):
    """
    y_true: 连续真实值
    y_pred: 连续预测值
    cutoff: 对 y_true 定义“是否爆发”的固定阈值

    逻辑：
    1. y_true >= cutoff -> 真实二分类标签
    2. 用 y_pred 作为连续得分，计算 ROC
    3. 用 Youden index = TPR - FPR 选最优阈值
    4. 用这个最优阈值把 y_pred 转成二分类
    5. 输出 precision / recall / f1 / accuracy / AUC / 最优阈值
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    # 1) 真实标签二值化
    y_true_bin = (y_true >= cutoff).astype(int)

    # 默认值
    best_cutoff_pred = np.nan
    auc = np.nan
    precision = np.nan
    recall = np.nan
    f1 = np.nan
    accuracy = np.nan
    outbreak_count_true = int(y_true_bin.sum())
    outbreak_count_pred = np.nan

    # 若真实标签只有一个类别，AUC/ROC 无法定义
    if len(np.unique(y_true_bin)) < 2:
        # 可以退化为直接用 cutoff 对 y_pred 二值化，方便至少给出部分指标
        y_pred_bin = (y_pred >= cutoff).astype(int)

        precision = precision_score(y_true_bin, y_pred_bin, zero_division=0)
        recall = recall_score(y_true_bin, y_pred_bin, zero_division=0)
        f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
        accuracy = accuracy_score(y_true_bin, y_pred_bin)
        outbreak_count_pred = int(y_pred_bin.sum())
        best_cutoff_pred = cutoff

        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "accuracy": accuracy,
            "auc": auc,
            "best_cutoff_pred": best_cutoff_pred,
            "outbreak_count_true": outbreak_count_true,
            "outbreak_count_pred": outbreak_count_pred
        }

    # 2) AUC
    auc = roc_auc_score(y_true_bin, y_pred)

    # 3) ROC + Youden index
    fpr, tpr, thresholds = roc_curve(y_true_bin, y_pred)
    youden_index = tpr - fpr
    best_idx = np.argmax(youden_index)
    best_cutoff_pred = thresholds[best_idx]

    # 4) 用最优阈值二值化预测
    y_pred_bin = (y_pred >= best_cutoff_pred).astype(int)

    # 5) 分类指标
    precision = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    recall = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
    accuracy = accuracy_score(y_true_bin, y_pred_bin)
    outbreak_count_pred = int(y_pred_bin.sum())

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "auc": auc,
        "best_cutoff_pred": best_cutoff_pred,
        "outbreak_count_true": outbreak_count_true,
        "outbreak_count_pred": outbreak_count_pred
    }

# ========== 6. 按省份评估 ==========
results = []

for city, group in df_test.groupby("City"):
    y_true = group["Labels"].values

    for model in model_cols:
        y_pred = group[model].values

        # 回归指标
        rmse, mae, mape = evaluate_regression(y_true, y_pred)

        # 爆发检测指标
        outbreak_metrics = evaluate_outbreak_detection(
            y_true, y_pred, cutoff=OUTBREAK_CUTOFF
        )


        results.append({
            "City": city,
            "Model": model,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "Precision_Outbreak": outbreak_metrics["precision"],
            "Recall_Outbreak": outbreak_metrics["recall"],
            "F1_Outbreak": outbreak_metrics["f1"],
            "Accuracy_Outbreak": outbreak_metrics["accuracy"],
            "AUC_Outbreak": outbreak_metrics["auc"],
            "Pred_Cutoff_Youden": outbreak_metrics["best_cutoff_pred"],
            "True_Outbreak_Count": outbreak_metrics["outbreak_count_true"],
            "Pred_Outbreak_Count": outbreak_metrics["outbreak_count_pred"]
        })

# 转为DataFrame
results_df = pd.DataFrame(results)

# ========== 7. 每个省份内排序 ==========
results_df["Rank_RMSE"] = results_df.groupby("City")["RMSE"].rank(method="min", ascending=True)
results_df["Rank_MAE"] = results_df.groupby("City")["MAE"].rank(method="min", ascending=True)
results_df["Rank_F1_Outbreak"] = results_df.groupby("City")["F1_Outbreak"].rank(method="min", ascending=False)
results_df["Rank_Recall_Outbreak"] = results_df.groupby("City")["Recall_Outbreak"].rank(method="min", ascending=False)

# ========== 8. 输出结果 ==========
print("\n=== 每个省份模型表现 ===")
print(results_df.sort_values(["City", "RMSE"]))

# 保存
results_df.to_csv("model_evaluation_by_city_with_outbreak.csv", index=False)

# ========== 9. 找每个省份最优模型 ==========
# 按 RMSE 最优
best_models_rmse = results_df.loc[
    results_df.groupby("City")["RMSE"].idxmin()
]

# 按爆发检测 F1 最优
best_models_f1 = results_df.loc[
    results_df.groupby("City")["F1_Outbreak"].idxmax()
]

print("\n=== 每个省份最优模型（按 RMSE） ===")
print(best_models_rmse[["City", "Model", "RMSE", "MAE", "F1_Outbreak", "Recall_Outbreak"]])

print("\n=== 每个省份最优模型（按爆发检测 F1） ===")
print(best_models_f1[["City", "Model", "F1_Outbreak", "Recall_Outbreak", "Precision_Outbreak", "RMSE"]])

# 如果你在 notebook 里，最后显示这个
results_df



=== 每个省份模型表现 ===
   City                Model        RMSE         MAE         MAPE  \
8    上海  XGBSingleForecaster    5.813703    2.528691   257.916199   
4    上海        MLPForecaster   13.489497   13.166873   940.638184   
0    上海    CNNLSTMForecaster   13.989664   11.334789   662.573669   
1    上海        CNNForecaster   14.060980   13.754335   981.298645   
6    上海        TCNForecaster   15.201476   14.850884  1067.083984   
..  ...                  ...         ...         ...          ...   
57   西藏       LSTMForecaster   19.525133   18.307835          NaN   
54   西藏    CNNLSTMForecaster   20.998292   18.770382          NaN   
59   西藏     ResNetForecaster   28.059396   24.031498          NaN   
56   西藏    DLinearForecaster   42.124916   33.318226          NaN   
62   西藏  XGBSingleForecaster  311.473050  100.092613          NaN   

    Precision_Outbreak  Recall_Outbreak  F1_Outbreak  Accuracy_Outbreak  \
8             0.478261         0.600000     0.532258           0.693122   
4  

,City,Model,RMSE,MAE,MAPE,Precision_Outbreak,Recall_Outbreak,F1_Outbreak,Accuracy_Outbreak,AUC_Outbreak,Pred_Cutoff_Youden,True_Outbreak_Count,Pred_Outbreak_Count,Rank_RMSE,Rank_MAE,Rank_F1_Outbreak,Rank_Recall_Outbreak
0,上海,CNNLSTMForecaster,13.989664,11.334789,662.573669,0.362205,0.836364,0.505495,0.523810,0.625237,-15.562588,55,127,3.0,2.0,4.0,1.0
1,上海,CNNForecaster,14.060980,13.754335,981.298645,0.000000,0.000000,0.000000,0.708995,0.474355,inf,55,0,4.0,4.0,8.0,8.0
2,上海,DLinearForecaster,24.722990,19.705896,1739.895264,0.483146,0.781818,0.597222,0.693122,0.741927,14.356506,55,89,8.0,8.0,1.0,2.0
3,上海,LSTMForecaster,17.058421,14.101822,819.995117,0.393258,0.636364,0.486111,0.608466,0.609905,-9.822233,55,89,7.0,5.0,6.0,5.0
4,上海,MLPForecaster,13.489497,13.166873,940.638184,0.293651,0.672727,0.408840,0.433862,0.491452,-12.542044,55,126,2.0,3.0,7.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,西藏,MLPForecaster,12.253623,12.246239,NaN,0.000000,0.000000,0.000000,1.000000,NaN,1.000000,0,0,1.0,1.0,1.0,1.0
59,西藏,ResNetForecaster,28.059396,24.031498,NaN,0.000000,0.000000,0.000000,0.740741,NaN,1.000000,0,49,7.0,7.0,1.0,1.0
60,西藏,TCNForecaster,14.013655,13.930249,NaN,0.000000,0.000000,0.000000,1.000000,NaN,1.000000,0,0,3.0,4.0,1.0,1.0
61,西藏,TransformerForecaster,14.129828,13.735126,NaN,0.000000,0.000000,0.000000,1.000000,NaN,1.000000,0,0,4.0,3.0,1.0,1.0


In [131]:
import pandas as pd

def summarize_metrics_iqr(results_df, metrics, index_col='City', model_col='Model', round_digits=3):
    """
    对多个指标输出:
    median (25%-75%)
    并按 median 排序

    参数
    ----
    results_df : DataFrame
    metrics : list[str]
        需要汇总的指标列名
    index_col : str
        行分组列，默认 City
    model_col : str
        模型列，默认 Model
    round_digits : int
        保留小数位数
    """
    summary_tables = {}

    for metric in metrics:
        pivot_df = results_df.pivot(index=index_col, columns=model_col, values=metric)

        stat_df = pivot_df.agg([
            'median',
            lambda x: x.quantile(0.25),
            lambda x: x.quantile(0.75)
        ]).T

        stat_df.columns = ['median', 'q25', 'q75']
        stat_df = stat_df.sort_values('median')

        stat_df[f'{metric}_summary'] = stat_df.apply(
            lambda row: f"{row['median']:.{round_digits}f} ({row['q25']:.{round_digits}f}-{row['q75']:.{round_digits}f})",
            axis=1
        )

        summary_tables[metric] = stat_df[[f'{metric}_summary']]

    return summary_tables


metrics = [
    # "RMSE",
    "MAE",
    # "MAPE",
    "Precision_Outbreak",
    "Recall_Outbreak",
    "F1_Outbreak",
    # "Accuracy_Outbreak",
    "AUC_Outbreak",
    "Pred_Cutoff_Youden",
    # "True_Outbreak_Count",
    # "Pred_Outbreak_Count"
]

summary_tables = summarize_metrics_iqr(results_df, metrics)
merged_summary = pd.concat(summary_tables.values(), axis=1)
merged_summary


/home/xutingfeng/miniforge3/envs/GEO/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4596: RuntimeWarning:

invalid value encountered in subtract

/home/xutingfeng/miniforge3/envs/GEO/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4596: RuntimeWarning:

invalid value encountered in subtract



,MAE_summary,Precision_Outbreak_summary,Recall_Outbreak_summary,F1_Outbreak_summary,AUC_Outbreak_summary,Pred_Cutoff_Youden_summary
Model,,,,,,
XGBSingleForecaster,2.275 (0.940-51.311),0.478 (0.079-0.584),0.720 (0.562-0.772),0.532 (0.142-0.611),0.684 (0.671-0.782),1.000 (0.496-1.431)
MLPForecaster,13.167 (12.302-13.363),0.294 (0.040-0.397),0.673 (0.372-0.686),0.331 (0.074-0.487),0.487 (0.463-0.506),-12.542 (-12.542--11.645)
CNNForecaster,13.754 (12.889-13.950),0.000 (0.000-0.020),0.000 (0.000-0.362),0.000 (0.000-0.038),0.482 (0.472-0.522),inf (-5.783-nan)
CNNLSTMForecaster,14.152 (10.871-19.481),0.352 (0.065-0.435),0.800 (0.697-0.848),0.478 (0.121-0.572),0.649 (0.572-0.680),-14.632 (-15.550--12.725)
LSTMForecaster,14.729 (13.615-17.402),0.393 (0.046-0.529),0.627 (0.373-0.756),0.400 (0.087-0.479),0.580 (0.543-0.603),-9.822 (-16.373--4.026)
TCNForecaster,14.851 (13.986-15.047),0.000 (0.000-0.020),0.000 (0.000-0.362),0.000 (0.000-0.038),0.482 (0.472-0.522),inf (-6.127-nan)
TransformerForecaster,14.902 (14.280-15.002),0.404 (0.116-0.467),0.655 (0.402-0.730),0.500 (0.186-0.542),0.623 (0.550-0.660),-13.541 (-14.494--10.721)
DLinearForecaster,19.790 (16.757-28.992),0.483 (0.076-0.520),0.733 (0.586-0.791),0.591 (0.133-0.600),0.739 (0.678-0.749),-1.894 (-6.484-1.402)
ResNetForecaster,30.056 (21.399-33.789),0.488 (0.064-0.553),0.667 (0.550-0.737),0.495 (0.118-0.595),0.673 (0.653-0.690),-18.313 (-26.243--5.618)


In [132]:
results_df.pivot(index='City', columns = 'Model',values='MAE').round(3)

Model,CNNForecaster,CNNLSTMForecaster,DLinearForecaster,LSTMForecaster,MLPForecaster,ResNetForecaster,TCNForecaster,TransformerForecaster,XGBSingleForecaster
City,,,,,,,,,
上海,13.754,11.335,19.706,14.102,13.167,30.815,14.851,14.959,2.529
内蒙古,12.860,20.192,13.808,16.496,12.273,36.763,13.957,15.044,0.453
北京,13.797,9.752,24.666,13.127,13.209,30.056,14.893,14.739,2.275
山东,14.104,10.406,19.790,12.296,13.516,13.628,15.200,14.902,1.426
山西,12.918,14.152,13.740,14.729,12.331,18.767,14.015,13.820,0.314
广东,349.072,348.604,289.658,336.211,348.484,408.418,350.168,348.400,308.028
西藏,12.834,18.770,33.318,18.308,12.246,24.031,13.930,13.735,100.093


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# 保证时间列是 datetime
plot_df = plot_df.copy()
plot_df["Time"] = pd.to_datetime(plot_df["Time"])

# 你这些模型的推荐配色
color_map = {
    "Ground truth": "#D62728",            # 高亮红色
    "CNNLSTMForecaster": "#1F77B4",       # 蓝
    "CNNForecaster": "#17BECF",           # 青
    "DLinearForecaster": "#2CA02C",       # 绿
    "LSTMForecaster": "#9467BD",          # 紫
    "MLPForecaster": "#FF7F0E",           # 橙
    "ResNetForecaster": "#8C564B",        # 棕
    "TCNForecaster": "#E377C2",           # 粉
    "TransformerForecaster": "#7F7F7F",   # 灰
    "XGBSingleForecaster": "#BCBD22",     # 黄绿
}

# 线型也稍微区分一下
dash_map = {
    "Ground truth": "solid",
    "CNNLSTMForecaster": "solid",
    "CNNForecaster": "dot",
    "DLinearForecaster": "dash",
    "LSTMForecaster": "longdash",
    "MLPForecaster": "dashdot",
    "ResNetForecaster": "solid",
    "TCNForecaster": "dot",
    "TransformerForecaster": "dash",
    "XGBSingleForecaster": "longdash",
}

# 图例顺序
model_order = [
    "Ground truth",
    "CNNLSTMForecaster",
    "CNNForecaster",
    "DLinearForecaster",
    "LSTMForecaster",
    "MLPForecaster",
    "ResNetForecaster",
    "TCNForecaster",
    "TransformerForecaster",
    "XGBSingleForecaster",
]

fig = go.Figure()

for model in model_order:
    sub_df = plot_df[plot_df["Model"] == model].sort_values("Time")
    if sub_df.empty:
        continue

    is_gt = model == "Ground truth"

    fig.add_trace(
        go.Scatter(
            x=sub_df["Time"],
            y=sub_df["Count"],
            mode="lines",
            name=model,
            line=dict(
                color=color_map.get(model, "#444444"),
                width=3.5 if is_gt else 1.8,
                dash=dash_map.get(model, "solid"),
            ),
            opacity=1.0 if is_gt else 0.9,
            hovertemplate=(
                f"<b>{model}</b><br>"
                "Time: %{x}<br>"
                "Count: %{y:.3f}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    template="plotly_white",
    width=1400,
    height=520,
    hovermode="x unified",
    legend=dict(
        title="Model",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
        bgcolor="rgba(255,255,255,0.85)",
    ),
    margin=dict(l=50, r=30, t=70, b=60),
)

fig.update_xaxes(
    title_text="Time",
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    rangeslider=dict(visible=True),   # 底部缩放条
    rangeselector=dict(               # 快速缩放按钮
        buttons=[
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=3, label="3m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            dict(count=1, label="1y", step="year", stepmode="backward"),
            dict(step="all", label="All"),
        ]
    ),
)

fig.update_yaxes(
    title_text="Count",
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    zeroline=False,
)

fig.show(config={
    "scrollZoom": True,   # 鼠标滚轮缩放
    "displaylogo": False,
    "doubleClick": "reset",
    "responsive": True,
})
